<a href="https://colab.research.google.com/github/pedroabsouza/projeto_desafios_tecnicos_ad/blob/main/desafio_01/Desafio1_API_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Qual objetivo deste Notebook?**

- Explicar o que é uma API e para que ela serve no trabalho de um analista de dados;

- Escolher um tema para a base de dados entre as oito opções que será utlizado nos próximos Desafios deste projeto;

- Conectar o Python à Google Sheets API (autenticar e escrever dados numa planilha);

- Aplicar o esquema de colunas do tema escolhido ao gerar dados mockados;

- Usar IA como apoio na geração de dados, deixando claro quando forem dados simulados e evitando apresentar estatísticas inventadas como resultados reais;

- Reconhecer os limites éticos do uso de IA na geração e no tratamento de dados.

## 🔑 **Pontos Chave**

- **API (Interface de Programação de Aplicações)** é um contrato entre dois sistemas: você pede algo num formato combinado e recebe a resposta no mesmo formato, sem precisar saber o que acontece por dentro. É a porta de entrada, não a casa inteira.

- A **Google Sheets API** é a porta que o Python usa para ler e escrever numa planilha do Google.

- A **API exige autenticação**: uma credencial que prova quem está pedindo o acesso. A planilha precisa ser compartilhada com ela; sem compartilhar, a escrita falha em silêncio.

- **Apps Script** roda dentro do Google, em JavaScript. O seu script Python roda fora, no Colab, e conversa com a planilha pela API. São ferramentas diferentes que se falam pela API. Não cole código Python no editor do Apps Script: não funciona.

- Bibliotecas como gspread ou google-api-python-client encapsulam a comunicação, para você não montar a chamada HTTP à mão.

- Cada linha escrita pela API é uma linha real, visível na hora, como se alguém tivesse digitado.

- Erros de permissão são o problema mais comum. Leia o erro retornado: quase sempre ele indica a causa.

- Toda requisição enviada a uma API recebe um **HTTP Status Code**, que indica o sucesso ou o insucesso da requisição.



#### Etapas feitas até agora:

> Instalação do Google Sheets API e Google Drive API nas bibliotecas do Google Cloud;

> IAM > 'Contas de Serviço' > Criação da Chave de Acesso 'desafio1-python'

> Criação da Planilha Desafio_01_Festival_Musica no Sheets. Compartilhar planilha com endereço de e-mail fornecedio na chave de acesso do Google Cloud como editor. Copiar o seu id (O que está entre d/ /edit no URL)

> No Colab, clicar na chave ao lado e inserir os nomes GCP_SA_JSON com o JSON da chave de Acesso e GS_ID com o ID da planilha.

# **Célula 01 · Instalar as bibliotecas gspread e faker**

> gspread: biblioteca Python que facilita o acesso ao Google Sheets pela API. Com poucas linhas, você abre uma planilha e lê ou escreve valores em células e linhas, sem montar as requisições HTTP manualmente.

> Faker: biblioteca que gera dados fictícios com aparência realista (nomes, endereços, e-mails, datas, textos etc.), inclusive em português do Brasil. É ela que permite montar uma base mockada (dados fictícios) sem usar dados de pessoas reais.

In [30]:
# Instala as duas bibliotecas externas usadas neste notebook:
#   gspread -> acessa e edita planilhas do Google Sheets pela API
#   faker   -> gera dados fictícios (nomes, datas, endereços etc.)
# O "!" executa o comando no terminal do Colab; o "-q" reduz as mensagens exibidas.
!pip install -q gspread faker

# **Célula 02 · Importações e Autenticação (service account)**

In [31]:
# Bibliotecas usadas ao longo do notebook.
import json, gspread, random, os, time
from faker import Faker
import pandas as pd

# date e timedelta são usados nas Células 05 e 08 para criar e somar datas.
from datetime import date, timedelta

from google.colab import files
# O userdata dá acesso às Secrets do Colab (credencial e ID da planilha).
from google.colab import userdata

# Lê o conteúdo do JSON da credencial (Secret GCP_SA_JSON) e converte o texto em dicionário.
info = json.loads(userdata.get("GCP_SA_JSON"))

# Cria o cliente do gspread autenticado com a service account.
gc = gspread.service_account_from_dict(info)

# Exibe o e-mail da credencial para confirmar que a autenticação funcionou.
print("Conectado como:", info["client_email"])

Conectado como: desafio1-python@probable-dream-515000-g8.iam.gserviceaccount.com


# **Célula 03 · Abrir a planilha**

In [32]:
# Lê o ID da planilha guardado na Secret GS_ID (copiado no Passo 5).
SHEET_ID = userdata.get("GS_ID")

# Define o nome que a aba da planilha vai receber.
NOME_TABELA = "atracoes"   # nome da tabela dimensão do seu tema (veja a tabela abaixo)

# Abre a planilha pelo ID e seleciona a primeira aba (ws será usada nas próximas células).
sh = gc.open_by_key(SHEET_ID)
ws = sh.sheet1

# Renomeia a aba com o nome da tabela e exibe os dados da planilha.
ws.update_title(NOME_TABELA)     # a aba passa a ter o nome da tabela
print(sh.title, "|", ws.title, "|", sh.url)

Desafio_01_Festival_Musica | atracoes | https://docs.google.com/spreadsheets/d/1rsGsQmq_8ISg4FifCGr6sjSUNbTOHmxk1HMs-UTj6aA


# **Preenchendo a Planilha com Dados Fictícios (Tabela Dimensão - Atração)**

### 04. Cabeçalho

In [33]:
# Substitua pelos nomes EXATOS de coluna da ficha do seu tema, na mesma ordem do plano.
# Esta lista define o cabeçalho e a ordem dos valores que serão enviados na Célula 05.
COLUNAS_DIMENSAO = ["atracao_id", "nome_artista", "genero_musical", "edicao_festival", "dia_apresentacao", "palco", "horario_inicio"]

# Escreve o cabeçalho a partir da célula A1.
# O parâmetro "values" recebe uma lista de linhas; como o cabeçalho é uma única linha, usamos [COLUNAS_DIMENSAO].
ws.update(range_name="A1", values=[COLUNAS_DIMENSAO])

# Lê a linha 1 de volta e compara com COLUNAS_DIMENSAO: True indica que o cabeçalho foi gravado corretamente.
print("Cabeçalho confere:", ws.row_values(1) == COLUNAS_DIMENSAO)

Cabeçalho confere: True


### 05. Inserir 10 linhas de teste

In [34]:
import random
from datetime import date, timedelta
from faker import Faker

# Cria o gerador do Faker em português do Brasil.
fake = Faker("pt_BR")

# Fixa as sementes para manter a reprodutibilidade.
Faker.seed(42)
random.seed(42)

# Valores exatos permitidos pela documentação da tabela
GENEROS = ["rock", "pop", "mpb", "eletronica", "funk", "sertanejo", "rap", "forró", "reggae"]
PALCOS = ["principal", "secundario", "eletronico", "acustico"]

# Edições no formato "Festival X AAAA" e suas respectivas janelas de 3 dias
DATAS_POR_EDICAO = {
    "Festival X 2023": [date(2023, 9, 15), date(2023, 9, 16), date(2023, 9, 17)],
    "Festival X 2024": [date(2024, 9, 13), date(2024, 9, 14), date(2024, 9, 15)],
    "Festival X 2025": [date(2025, 9, 19), date(2025, 9, 20), date(2025, 9, 21)],
}

# Horários programados no formato HH:MM
HORARIOS_SHOW = [
    "14:00", "15:15", "16:30", "17:45",
    "19:00", "20:15", "21:30", "23:00", "00:30"
]

def gerar_nome_artista():
    estilo = random.choice(["solo", "banda", "dj"])
    if estilo == "solo":
        return fake.name()
    elif estilo == "banda":
        return f"Os {fake.last_name()}s" if random.random() > 0.5 else f"Banda {fake.word().capitalize()}"
    else:
        return f"DJ {fake.first_name()}"

def gerar_linha(i):
    # Sorteia a edição e uma data dentro da janela de 3 dias correspondente
    edicao = random.choice(list(DATAS_POR_EDICAO.keys()))
    dia_show = random.choice(DATAS_POR_EDICAO[edicao])

    # Sorteia o gênero e associa um palco coerente com os valores permitidos
    genero = random.choice(GENEROS)
    if genero == "eletronica":
        palco = random.choices(["eletronico", "principal"], weights=[85, 15])[0]
    elif genero in ["mpb", "reggae"]:
        palco = random.choices(["acustico", "secundario", "principal"], weights=[50, 30, 20])[0]
    else:
        palco = random.choice(["principal", "secundario"])

    return {
        "atracao_id": i,                               # inteiro (PK)
        "nome_artista": gerar_nome_artista(),          # texto
        "genero_musical": genero,                      # texto (domínio exato em minúsculas)
        "edicao_festival": edicao,                     # texto (padrão "Festival X AAAA")
        "dia_apresentacao": dia_show.isoformat(),      # data (AAAA-MM-DD dentro da janela de 3 dias)
        "palco": palco,                                # texto (domínio exato em minúsculas/sem acento)
        "horario_inicio": random.choice(HORARIOS_SHOW) # texto (HH:MM)
    }

# Chama a função 10 vezes (i = 1, 2, ..., 10)
linhas = [gerar_linha(i) for i in range(1, 11)]

# Converte para a ordem de COLUNAS_DIMENSAO
valores = [[linha[c] for c in COLUNAS_DIMENSAO] for linha in linhas]

# Envia para a planilha
ws.append_rows(valores, value_input_option="RAW")
print(len(valores), "linhas enviadas")

10 linhas enviadas


### 06. Conferência

In [35]:
# Lê todos os valores da aba (lista de linhas) para conferir o que realmente chegou à planilha.
valores_lidos = ws.get_all_values()

# Quantidade de linhas: cabeçalho + 10 linhas de dados.
print("Linhas (cabeçalho + dados):", len(valores_lidos))    # esperado: 11

# A primeira linha lida (índice 0) deve ser igual à lista COLUNAS_DIMENSAO.
print("Cabeçalho confere:", valores_lidos[0] == COLUNAS_DIMENSAO)

# Mostra as 3 primeiras linhas (cabeçalho + 2 registros) para uma conferência rápida.
print(valores_lidos[:3])

Linhas (cabeçalho + dados): 21
Cabeçalho confere: True
[['atracao_id', 'nome_artista', 'genero_musical', 'edicao_festival', 'dia_apresentacao', 'palco', 'horario_inicio'], ['1', 'Brenda Alves', 'rock', 'Festival X 2025', '2025-09-19', 'secundario', '17:45'], ['2', 'DJ Yan', 'pop', 'Festival X 2023', '2023-09-17', 'principal', '21:30']]


# **Trabalhando agora com a Tabela Fato - Ingressos_Vendidos**


### 07. Definir o Esquema

In [36]:
# Nome da tabela fato do seu tema (veja a tabela acima). Ele será o nome da segunda guia da planilha.
NOME_TABELA_FATO = "ingressos_vendidos"

# Substitua pelos nomes EXATOS de coluna da tabela fato, na mesma ordem do plano.
COLUNAS_FATO = ["venda_id", "atracao_id", "tipo_ingresso", "canal_venda", "data_hora_venda", "valor", "forma_pagamento"]

# Nome da coluna da tabela fato que é a chave estrangeira (a que guarda o ID da tabela dimensão).
COLUNA_FK = "atracao_id"

# Confere se a chave estrangeira é mesmo uma das colunas da tabela fato.
assert COLUNA_FK in COLUNAS_FATO, "COLUNA_FK precisa ser um dos nomes de COLUNAS_FATO"
print(NOME_TABELA_FATO, "|", COLUNAS_FATO)

ingressos_vendidos | ['venda_id', 'atracao_id', 'tipo_ingresso', 'canal_venda', 'data_hora_venda', 'valor', 'forma_pagamento']


### 08. Montar a Estrutura

In [37]:
import random
from datetime import date, datetime, timedelta
from faker import Faker

# Cria o gerador do Faker em português do Brasil.
fake = Faker("pt_BR")

# Fixa as sementes para manter a reprodutibilidade.
# Dica: se quiser que os sorteios da tabela fato sejam diferentes dos da dimensão,
# você pode usar outra seed aqui, como 100.
Faker.seed(100)
random.seed(100)

# Domínios exatos especificados na documentação da Tabela 2
TIPOS_INGRESSO = ["pista", "pista_premium", "vip", "camarote"]
CANAIS_VENDA = ["site", "app", "bilheteria_fisica", "revenda_parceira"]
FORMAS_PAGAMENTO = ["cartao_credito", "cartao_debito", "pix", "boleto"]

# Faixas de preço coerentes (em reais) para cada setor/tipo de ingresso
PRECOS_POR_TIPO = {
    "pista": (180.00, 320.00),
    "pista_premium": (350.00, 580.00),
    "vip": (650.00, 950.00),
    "camarote": (1100.00, 1800.00),
}

def gerar_linha_fato(i, ids_dim):
    # Sorteia o tipo de ingresso com pesos realistas (mais vendas de pista que camarote)
    tipo = random.choices(TIPOS_INGRESSO, weights=[50, 25, 15, 10])[0]

    # Calcula um valor decimal coerente com o tipo de ingresso sorteado
    preco_min, preco_max = PRECOS_POR_TIPO[tipo]
    valor_ingresso = round(random.uniform(preco_min, preco_max), 2)

    # Sorteia o canal e ajusta a forma de pagamento de forma lógica
    canal = random.choices(CANAIS_VENDA, weights=[45, 35, 10, 10])[0]
    if canal == "bilheteria_fisica":
        # Na bilheteria presencial não faz sentido pagar com boleto
        pagamento = random.choices(["cartao_credito", "cartao_debito", "pix"], weights=[45, 30, 25])[0]
    else:
        pagamento = random.choices(FORMAS_PAGAMENTO, weights=[50, 10, 30, 10])[0]

    # Gera data e hora da venda no formato padrão AAAA-MM-DD HH:MM:SS
    momento_venda = fake.date_time_between(
        start_date=datetime(2023, 6, 1, 8, 0, 0),
        end_date=datetime(2025, 9, 21, 22, 0, 0)
    ).strftime("%Y-%m-%d %H:%M:%S")

    return {
        "venda_id": i,                                   # inteiro (PK)
        "atracao_id": random.randint(1, 10),             # inteiro (FK -> atracoes.atracao_id de 1 a 10)
        "tipo_ingresso": tipo,                           # texto ("pista", "pista_premium", "vip", "camarote")
        "canal_venda": canal,                            # texto ("site", "app", "bilheteria_fisica", "revenda_parceira")
        "data_hora_venda": momento_venda,                # data/hora ("AAAA-MM-DD HH:MM:SS")
        "valor": valor_ingresso,                         # decimal (valor em reais com 2 casas)
        "forma_pagamento": pagamento,                    # texto ("cartao_credito", "cartao_debito", "pix", "boleto")
    }

# **Célula 09 - Gerar as Tabelas**




In [38]:
# Quantidade de linhas de cada tabela. O total precisa ser de pelo menos 20.000.
N_DIM = 10_000
N_FATO = 20_000

# Reinicia as sementes (como na Célula 05): o resultado fica reproduzível.
Faker.seed(42)
random.seed(42)

# ---- Tabela dimensão: usa a função gerar_linha (Célula 05) ----
# gerar_linha devolve um dicionário por linha; o DataFrame organiza as colunas na ordem de COLUNAS_DIMENSAO.
tabela_dim = pd.DataFrame([gerar_linha(i) for i in range(1, N_DIM + 1)], columns=COLUNAS_DIMENSAO)

# ---- Tabela fato: usa a função gerar_linha_fato (Célula 08) ----
# ids_dim é a lista de IDs da dimensão GERADA acima: é nela que a fato sorteia a chave estrangeira.
ids_dim = list(tabela_dim[COLUNAS_DIMENSAO[0]])
tabela_fato = pd.DataFrame([gerar_linha_fato(i, ids_dim) for i in range(1, N_FATO + 1)], columns=COLUNAS_FATO)

# Guarda as tabelas usando o nome de cada uma (NOME_TABELA e NOME_TABELA_FATO).
BASES_GERADAS = {NOME_TABELA: tabela_dim, NOME_TABELA_FATO: tabela_fato}

# Exibe a quantidade de linhas geradas e o total.
print({nome: len(df) for nome, df in BASES_GERADAS.items()}, "| total:", len(tabela_dim) + len(tabela_fato))

{'atracoes': 10000, 'ingressos_vendidos': 20000} | total: 30000


# **Célula 10 - Validar e gerar os arquivos CSV**

In [39]:
# Confere se as colunas de cada tabela seguem o esquema da ficha (Células 04 e 07).
assert list(tabela_dim.columns) == COLUNAS_DIMENSAO, "Colunas diferentes do esquema na tabela dimensão"
assert list(tabela_fato.columns) == COLUNAS_FATO, "Colunas diferentes do esquema na tabela fato"

# Confere o ID único na dimensão, a chave estrangeira na fato e o volume mínimo.
assert tabela_dim[COLUNAS_DIMENSAO[0]].is_unique, "ID repetido na tabela dimensão"
assert tabela_fato[COLUNA_FK].isin(tabela_dim[COLUNAS_DIMENSAO[0]]).all(), "Chave estrangeira sem ID na dimensão"
assert len(tabela_dim) + len(tabela_fato) >= 20_000, "Abaixo do mínimo (20.000 linhas somadas)."

# Para cada tabela: grava o CSV, lê de volta para conferir e baixa o arquivo.
os.makedirs("bases_geradas", exist_ok=True)
for nome, df in BASES_GERADAS.items():
    caminho = f"bases_geradas/{nome}.csv"
    df.to_csv(caminho, index=False, encoding="utf-8")
    lido = pd.read_csv(caminho)
    assert len(lido) == len(df) and list(lido.columns) == list(df.columns), f"CSV diferente em {nome}"
    print(nome + ".csv ->", len(lido), "linhas")
    files.download(caminho)

atracoes.csv -> 10000 linhas


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ingressos_vendidos.csv -> 20000 linhas


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Célula 11 - Função para envio de dados da Planilha**

In [40]:
def enviar(sh, df, nome_aba, lote=5000, pausa=2):
    # Reaproveita a aba se ela já existir (limpando o conteúdo) ou cria uma nova.
    try:
        aba = sh.worksheet(nome_aba)
        aba.clear()
    except gspread.WorksheetNotFound:
        aba = sh.add_worksheet(title=nome_aba, rows=len(df) + 1, cols=len(df.columns))

    # Ajusta o tamanho da aba e grava o cabeçalho na linha 1.
    aba.resize(rows=len(df) + 1, cols=len(df.columns))
    aba.update(range_name="A1", values=[list(df.columns)])

    # Converte a tabela em lista de linhas (valor ausente vira texto vazio).
    valores = df.astype(object).where(df.notna(), "").values.tolist()

    # Envia em lotes, com pausa para respeitar o limite de escritas por minuto.
    for i in range(0, len(valores), lote):
        aba.append_rows(valores[i:i + lote], value_input_option="RAW")
        print(f"  {nome_aba}: {min(i + lote, len(valores))} / {len(valores)}")
        time.sleep(pausa)

    # Lê de volta e confere a quantidade de linhas (sem contar o cabeçalho).
    lidas = len(aba.col_values(1)) - 1
    if lidas != len(df):
        raise RuntimeError(f"Aba {nome_aba}: enviei {len(df)}, li {lidas}.")
    print(f"OK {nome_aba}: {len(df)} linhas na planilha")

# **Célula 12 - Enviar as Tabelas Geradas**

In [41]:
# Envia cada tabela gerada para a aba de mesmo nome (dimensão e fato).
for nome, df in BASES_GERADAS.items():
    enviar(sh, df, nome)

  atracoes: 5000 / 10000
  atracoes: 10000 / 10000
OK atracoes: 10000 linhas na planilha
  ingressos_vendidos: 5000 / 20000
  ingressos_vendidos: 10000 / 20000
  ingressos_vendidos: 15000 / 20000
  ingressos_vendidos: 20000 / 20000
OK ingressos_vendidos: 20000 linhas na planilha


# **Célula 13 - Validar**

In [42]:
# Soma e exibe a quantidade total de linhas das tabelas.
total = sum(len(df) for df in BASES_GERADAS.values())
print("Total de linhas:", total)
# Interrompe a execução se o volume mínimo não for atingido.
assert total >= 20_000, "Abaixo do mínimo (20.000 linhas somadas)."

# Confere os cabeçalhos e a quantidade de registros em cada aba.
for nome, df in BASES_GERADAS.items():
    w = sh.worksheet(nome)
    assert w.row_values(1) == list(df.columns), f"Cabeçalho diferente em {nome}"
    assert len(w.col_values(1)) - 1 == len(df), f"Contagem diferente em {nome}"

# Confere o relacionamento: toda chave estrangeira da tabela fato existe na tabela dimensão.
fk_fato = BASES_GERADAS[NOME_TABELA_FATO][COLUNA_FK]
ids_dim = BASES_GERADAS[NOME_TABELA][COLUNAS_DIMENSAO[0]]
assert fk_fato.isin(ids_dim).all(), "Há chave estrangeira sem ID correspondente na tabela dimensão"

# Confirma que todas as verificações passaram.
print("A planilha confere com a base gerada.")

Total de linhas: 30000
A planilha confere com a base gerada.
